In [199]:
import numpy as np
import csv,os
from openai import OpenAI
from sklearn.metrics.pairwise import cosine_similarity

#import data from CSVs
data = []
files=["../2_Database/TopBeerData.csv","../2_Database/GoodBeerData.csv","../2_Database/RestBeerData.csv"]
for i in range (0,len(files)):
    with open(files[i],"r",encoding="utf-8") as file:
        reader = csv.reader(file)
        next(reader) # remove header
        for row in reader:
            data.append(row)
print(len(data))
with open("BeerData.csv", 'w', newline='',encoding='utf-8') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(['beer_text','brewery','beer'])
textual_data = []
with open("BeerData.csv", "a", newline="", encoding="utf-8") as csvfile:
    writer = csv.writer(csvfile)
    for row in data[1:]:
        beer_text = (
#            f"brewery: {row[2]}\n" #brewery removed since adds bias
            f"beer name: {row[3]} |"
            f"beer kind: {row[4]} |"
            f"description: {row[11]}|"
            f"ABV: {row[6]} |"
            f"IBU: {row[7]} |"
            f"rating: {row[8]}"
        )
        writer.writerow([beer_text,row[2],row[3]])
        textual_data.append(beer_text)
print("Adatok kiíratva CSV-be.",len(beer_text))

2303
Adatok kiíratva CSV-be. 92


In [200]:
# ---------------------------------------------------
# generate embeddings with OpenAI - COSTS MONEY
# ---------------------------------------------------

client = OpenAI(
    api_key=os.environ.get("OPENAI_API_KEY")
)

embeddings = []
batch_size = 100
for i in range(0, len(textual_data), batch_size):
    batch = textual_data[i:i + batch_size]
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=batch
    )
    embeddings.extend(
        item.embedding
        for item in response.data
    )

print(len(embeddings))

X_emb = np.array(embeddings)
print(X_emb.shape)

with open("BeerEmbeddings.csv", 'w', newline='',encoding='utf-8') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow([""] * 1536) #1536 nr of coloumns  
    for row in X_emb:
        writer.writerow(row)
print("Embedding kiíratva CSVbe.")

2302
(2302, 1536)
Embedding kiíratva CSVbe.


In [201]:
from difflib import SequenceMatcher

beer_data=[]
with open("BeerData.csv","r",encoding="utf-8") as file:
    reader = csv.reader(file)
    next(reader) # remove header
    for row in reader:
        beer_data.append(row)
print(len(beer_data))

#get user beer preference
pref_beer = input("Add meg a sör nevét: ")

count=0
for i, text in enumerate(beer_data):
    if pref_beer in text[2]:
        print(i,text[2],"Full matching of the beer name!!! \n")
        count=1
        break
    else:
        pass #do nothing

best_sim=0
for i, text in enumerate(beer_data):
    sim = SequenceMatcher(None, text[2], pref_beer).ratio()  
    if sim > best_sim:
        best_sim = sim
        best_idx = i
        best_text=text[2]

if (count==0 and best_sim<0.6) :     #0.6 is arbitrary
    print("nem találtam a kiválasztott sört")
else:
    print(f"Index: {best_idx}")
    print(f"Similarity: {best_sim:.3f}")
    print(f"A kiválasztott sör neve: {best_text}")
    pref_beer=best_text

2302


Add meg a sör nevét:  Citra Triple IPA


1038 Citra Triple IPA Full matching of the beer name!!! 

Index: 1038
Similarity: 1.000
A kiválasztott sör neve: Citra Triple IPA


In [211]:
#CSV reading
emb_data=[]
with open("BeerEmbeddings.csv","r",encoding="utf-8") as file:
    reader = csv.reader(file)
    next(reader) # remove header
    for row in reader:
        emb_data.append(row)
    
    #cosine similarity
    emb_data = np.array(emb_data, dtype=np.float32) #necessary conversion
    
    pref = emb_data[best_idx].reshape(1, -1)
    similarities = cosine_similarity(pref, emb_data)[0]
    top_k = 6
    top_idx = np.argsort(similarities)[::-1] 
    top_idx = top_idx[top_idx != (best_idx)][:top_k]
    print(top_idx)
    for i in top_idx:
        print(i, similarities[i],beer_data[i])
        print("\n")


[ 190 1035  927 1379 1021  193]
190 0.82407963 ['beer name: Beer Lovers Only |beer kind: IPA - Triple |description: Simcoe, Citra, Strata, Citra Cryo |ABV: 9.0 |IBU: N/A |rating: 3.88', 'Brewing Vibes Brewery', 'Beer Lovers Only']


1035 0.8102739 ['beer name: Triple Flower Power 2023 |beer kind: IPA - Triple |description:  |ABV: 9.0 |IBU: 60 |rating: 3.77', 'FIRST Craft Beer', 'Triple Flower Power 2023']


927 0.80952156 ['beer name: Red IPA |beer kind: IPA - Red |description:  |ABV: 5.0 |IBU: N/A |rating: 3.57', 'Tinta Brewing Co.', 'Red IPA']


1379 0.8000183 ['beer name: Yellow Haze - Citra |beer kind: Pale Ale - American |description:  |ABV: 5.5 |IBU: N/A |rating: 3.81', 'Brew Your Mind', 'Yellow Haze - Citra']


1021 0.7969059 ['beer name: Sour Series - Sour Citra IPA |beer kind: IPA - Sour |description: Kettlesoured Singlehop IPA with the most suited hop for this style, the Citra. If you need a refreshing IPA this is the one. |ABV: 6.5 |IBU: 18 |rating: 3.67', 'HORIZONT Brewing'